# 02 - Exploratory Data Analysis (EDA)
Analisis mendalam terhadap dataset GDP Growth Indonesia.

# 02 - Exploratory Data Analysis (EDA) - Regional ASEAN
Analisis mendalam terhadap dataset finansial dan demografi antar negara ASEAN.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('../data/eda_outputs', exist_ok=True)
df = pd.read_csv('../data/processed/dataset_asean.csv')
print('Shape data ASEAN:', df.shape)

print("\n" + "="*40)
print("       OVERVIEW DATASET ASEAN       ")
print("="*40)
print("1. Daftar Negara ASEAN di Dataset:")
print(df['country'].unique())
print("-" * 40)
print("2. Jumlah Baris Data per Negara:")
print(df['country'].value_counts())
print("-" * 40)
print(f"3. Rentang Tahun Data: {df['year'].min()} sampai {df['year'].max()}")
print("="*40)

df.head()

In [ ]:
# 2. Heatmap Korelasi Fitur ASEAN Baru
fitur_asean_baru = ['GDP_Growth', 'GDP_Per_Capita', 'Population_Growth', 'Exports_pct', 'Imports_pct', 'Life_Expectancy']
fitur_tersedia = [col for col in fitur_asean_baru if col in df.columns]

corr = df[fitur_tersedia].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr, 
    mask=mask, 
    annot=True, 
    fmt='.2f', 
    cmap='RdBu_r',
    center=0, 
    ax=ax, 
    square=True, 
    linewidths=0.5
)

ax.set_title('Heatmap Korelasi Antar Variabel ASEAN', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

if 'GDP_Growth' in corr.columns:
    print('Korelasi dengan GDP_Growth (Fitur Baru ASEAN):')
    print(corr['GDP_Growth'].drop('GDP_Growth').sort_values(ascending=False))

In [ ]:
# 3. Scatter plots tiap variabel baru vs GDP Growth
fitur_scatter = ['GDP_Per_Capita', 'Life_Expectancy', 'Exports_pct', 'Imports_pct']
fitur_scatter_tersedia = [col for col in fitur_scatter if col in df.columns]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, feat in enumerate(fitur_scatter_tersedia):
    sns.scatterplot(
        data=df,
        x=feat,
        y='GDP_Growth',
        hue='country',
        alpha=0.8,
        s=60,
        ax=axes[i]
    )

    df_clean = df[[feat, 'GDP_Growth']].dropna()
    if not df_clean.empty:
        z = np.polyfit(df_clean[feat], df_clean['GDP_Growth'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(df_clean[feat].min(), df_clean[feat].max(), 100)
        axes[i].plot(x_line, p(x_line), 'r--', alpha=0.8, label='Garis Tren Global')
    
    axes[i].set_xlabel(feat, fontsize=11)
    axes[i].set_ylabel('GDP Growth (%)', fontsize=11)
    axes[i].set_title(f'{feat} vs GDP Growth', fontsize=12, fontweight='bold')
    axes[i].legend(fontsize=9, loc='best')

plt.suptitle('Scatter Plot: Fitur ASEAN Baru vs GDP Growth', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Distribusi dan Deteksi Outlier GDP Growth Per Negara menggunakan Box Plot
plt.figure(figsize=(14, 7))
sns.boxplot(
    data=df, 
    x='country', 
    y='GDP_Growth', 
    palette='Set2'
)

sns.stripplot(
    data=df, 
    x='country', 
    y='GDP_Growth', 
    color='black', 
    alpha=0.3, 
    size=4, 
    jitter=True
)

plt.axhline(0, color='red', linestyle='--', alpha=0.5)
plt.title('Analisis Distribusi dan Outlier GDP Growth per Negara ASEAN', fontsize=14, fontweight='bold')
plt.xlabel('Negara')
plt.ylabel('GDP Growth (%)')
plt.xticks(rotation=30)
plt.grid(axis='y', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# 5. Statistik deskriptif lengkap per negara
print("=== STATISTIK DESKRIPTIF AGREGAT TOTAL ===")
desc_total = df.describe().round(3)
display(desc_total)

print("\n" + "="*60 + "\n")

print("=== STATISTIK RATA-RATA UTAMA PER NEGARA ===")
fitur_stats = ['GDP_Growth', 'GDP_Per_Capita', 'Population_Growth', 'Exports_pct', 'Imports_pct', 'Life_Expectancy']
fitur_stats_tersedia = [col for col in fitur_stats if col in df.columns]

desc_per_country = df.groupby('country')[fitur_stats_tersedia].mean().round(3)
display(desc_per_country)

desc_per_country.to_csv('../data/eda_outputs/descriptive_stats.csv')
print("\n[INFO] File deskriptif statistik per negara berhasil diperbarui di '../data/eda_outputs/descriptive_stats.csv'")